# Role-Aware SAAMR: SDF to OpenFF/OpenMM

**Author:** Joseph R. Laforet Jr.

This notebook starts from SDF files generated by `Role_Aware_SAAMR_Quickstart.ipynb` and demonstrates the downstream handoff:

```text
Primitive -> RDKit -> SDF -> OpenFF -> OpenMM
```

Generated simulation artifacts are written under `examples_system/role_aware_saamr_outputs/` and are ignored by git.

## 1. Environment and Inputs

In [ ]:
from pathlib import Path
import sys


def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")


EXAMPLES_ROOT = find_examples_root()
LOCAL_MUPT_SOURCE = EXAMPLES_ROOT / "mupt"
if LOCAL_MUPT_SOURCE.exists():
    sys.path.insert(0, str(LOCAL_MUPT_SOURCE))

SDF_DIR = EXAMPLES_ROOT / "examples_system" / "role_aware_saamr_outputs" / "sdf"
SIM_DIR = EXAMPLES_ROOT / "examples_system" / "role_aware_saamr_outputs" / "openmm"
SIM_DIR.mkdir(parents=True, exist_ok=True)

print(f"SDF directory: {SDF_DIR}")
print(f"Simulation output directory: {SIM_DIR}")

## 2. Load SDF Files with RDKit

In [ ]:
from rdkit import Chem

sdf_paths = sorted(SDF_DIR.glob("psu_pes_chain_*.sdf"))
if not sdf_paths:
    raise FileNotFoundError(
        f"No SDF files found in {SDF_DIR}. Run Role_Aware_SAAMR_Quickstart.ipynb first."
    )

rdkit_mols = []
for path in sdf_paths:
    supplier = Chem.SDMolSupplier(str(path), removeHs=False, sanitize=False)
    mol = supplier[0]
    if mol is None:
        raise ValueError(f"Could not read {path}")
    Chem.SanitizeMol(Chem.Mol(mol))
    rdkit_mols.append(mol)

print(f"Loaded {len(rdkit_mols)} SDF molecule(s)")
for path, mol in zip(sdf_paths, rdkit_mols):
    print(f"  {path.name}: atoms={mol.GetNumAtoms()}, bonds={mol.GetNumBonds()}")

## 3. Reconstruct MuPT SAAMR Hierarchies

This confirms that SDF files still contain enough information to recover the role-aware MuPT hierarchy.

In [ ]:
from mupt.interfaces.rdkit import primitive_from_rdkit
from mupt.roles import PrimitiveRole

reconstructed = [primitive_from_rdkit(mol, denest=False) for mol in rdkit_mols]

for idx, primitive in enumerate(reconstructed):
    assert primitive.role == PrimitiveRole.UNIVERSE
    assert len(primitive.children) == 1
    assert primitive.children[0].role == PrimitiveRole.SEGMENT
    assert all(residue.role == PrimitiveRole.RESIDUE for residue in primitive.children[0].children)
    print(
        f"reconstructed {idx}: residues={len(primitive.children[0].children)}, "
        f"particles={len(primitive.leaves)}"
    )

## 4. OpenFF Availability

The cells below are dependency-aware. If `openff-toolkit` is unavailable, the notebook reports the missing dependency and skips parameterization. Set the run flags to `True` when using an environment with OpenFF installed.

In [ ]:
try:
    from openff.toolkit import ForceField, Molecule, Topology
    from openff.units import unit
    OPENFF_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENFF_AVAILABLE = False
    OPENFF_IMPORT_ERROR = exc

try:
    import openmm
    from openmm import unit as omm_unit
    OPENMM_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENMM_AVAILABLE = False
    OPENMM_IMPORT_ERROR = exc

RUN_OPENFF_PARAMETERIZATION = False
RUN_OPENMM_MINIMIZATION = False

print(f"OpenFF available: {OPENFF_AVAILABLE}")
if not OPENFF_AVAILABLE:
    print(f"  {OPENFF_IMPORT_ERROR}")
print(f"OpenMM available: {OPENMM_AVAILABLE}")
if not OPENMM_AVAILABLE:
    print(f"  {OPENMM_IMPORT_ERROR}")

## 5. Convert RDKit Molecules to OpenFF Molecules

In [ ]:
off_molecules = []
if OPENFF_AVAILABLE:
    for mol in rdkit_mols:
        off_mol = Molecule.from_rdkit(
            mol,
            allow_undefined_stereo=True,
            hydrogens_are_explicit=True,
        )
        off_molecules.append(off_mol)
    print(f"Created {len(off_molecules)} OpenFF Molecule object(s)")
else:
    print("Skipping OpenFF conversion because openff-toolkit is not installed.")

## 6. Optional OpenFF Parameterization

This cell is intentionally off by default because partial-charge assignment and parameterization can be slow. Enable it in a properly configured OpenFF environment.

In [ ]:
interchange = None
if OPENFF_AVAILABLE and RUN_OPENFF_PARAMETERIZATION:
    ff = ForceField("openff-2.2.1.offxml")
    topology = Topology.from_molecules(off_molecules)
    interchange = ff.create_interchange(topology)
    print(f"Parameterized topology with {interchange.topology.n_atoms} atoms")
else:
    print("OpenFF parameterization skipped. Set RUN_OPENFF_PARAMETERIZATION = True to run it.")

## 7. Optional OpenMM Minimization

This cell runs only if OpenFF parameterization produced an `Interchange` and OpenMM is available.

In [ ]:
if interchange is not None and OPENMM_AVAILABLE and RUN_OPENMM_MINIMIZATION:
    system = interchange.to_openmm(combine_nonbonded_forces=True)
    topology = interchange.to_openmm_topology()
    positions = interchange.positions.to_openmm()
    integrator = openmm.LangevinMiddleIntegrator(
        300 * omm_unit.kelvin,
        1 / omm_unit.picosecond,
        1 * omm_unit.femtosecond,
    )
    simulation = openmm.app.Simulation(topology, system, integrator)
    simulation.context.setPositions(positions)
    simulation.minimizeEnergy(maxIterations=25)
    state = simulation.context.getState(getEnergy=True)
    print(f"Minimized potential energy: {state.getPotentialEnergy()}")
else:
    print("OpenMM minimization skipped. Enable both OpenFF parameterization and OpenMM minimization to run it.")

## 8. Optional Export Templates

These cells are templates for downstream exporters. They are disabled by default so tutorial execution does not create heavy MD artifacts.

In [ ]:
RUN_GROMACS_EXPORT = False
RUN_LAMMPS_EXPORT = False

if interchange is not None and RUN_GROMACS_EXPORT:
    gromacs_prefix = SIM_DIR / "role_aware_saamr"
    interchange.to_gromacs(str(gromacs_prefix), decimal=5)
    print(f"Wrote GROMACS files with prefix {gromacs_prefix}")
else:
    print("GROMACS export skipped.")

if interchange is not None and RUN_LAMMPS_EXPORT:
    print("LAMMPS export hook goes here once a project-standard exporter is selected.")
else:
    print("LAMMPS export skipped.")

## 9. Summary

This notebook is the downstream half of the workflow. The first notebook builds role-aware MuPT SAAMR systems and writes SDF files; this notebook loads those files and provides the OpenFF/OpenMM handoff path used for simulation setup.